# A4 – Oblikovanje značilk

**Vodena demonstracija · Olist**

## Priprava

Notebook je samostojen in deluje lokalno ter v Google Colabu. Celice izvajaj od
zgoraj navzdol s **Shift + Enter**. Ob menjavi stolpcev ponovi ustrezne naslednje celice.

Prva celica poišče `utils/orodja.py`; v Colabu po potrebi prenese repozitorij in namesti
knjižnice iz `requirements.txt`. Lokalno jih enkrat namestimo po navodilih v README.
Druga celica pokliče skupni `poskrbi_za_podatke()` in vrne mapo `DATA`.
Funkcija uporabi pripravljene ali že prenesene podatke; manjkajoče prenese s Kaggla.
CSV-je nato beremo iz `DATA`, ne glede na mapo, v kateri je zvezek odprt.
Izvornih vrednosti ne popravljamo in rezultatov vaj ne zapisujemo samodejno v datoteke.

Najprej oblikujemo ponovljive značilke, njihovo uporabnost pa pozneje preverimo z validacijo modela.


In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not (KOREN / "utils/orodja.py").is_file():
        if KOREN.exists():
            raise FileNotFoundError(f"Mapa {KOREN} že obstaja, vendar v njej ni utils/orodja.py.")
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(KOREN / "requirements.txt")],
        check=True,
    )
else:
    KOREN = Path.cwd().resolve()
    while not (KOREN / "utils/orodja.py").is_file() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
    if not (KOREN / "utils/orodja.py").is_file():
        raise FileNotFoundError("Zaženite zvezek v prenesenem repozitoriju ali njegovi podmapi (manjka utils/orodja.py).")

if str(KOREN) not in sys.path:
    sys.path.insert(0, str(KOREN))
print("Gradivo je v:", KOREN)

### Skupni podatkovni vhod
`poskrbi_za_podatke()` iz `utils/orodja.py` pripravi skupno mapo podatkov in jo vrne kot `DATA`.

Izvirne tabele Olista beremo iz `DATA / "trgovina"`; raven opazovanja pripravimo v nadaljevanju.


In [ ]:
from utils.orodja import poskrbi_za_podatke

DATA = poskrbi_za_podatke()

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

pd.set_option("display.max_columns", 15)
pd.set_option("display.max_colwidth", 35)
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11})

## 1. Raven podatkov in ključi

`order_id` označuje naročilo; postavke in plačila so lahko več vrstic na naročilo.
`customer_id` je vezan na posamezno naročilo, **`customer_unique_id`** pa poveže
ponovne nakupe istega kupca. ID-ji so ključi, ne merjene značilke.

Vrednost naročila bo vsota cen postavk **brez poštnine** v BRL. Plačila agregiramo
ločeno; ne predpostavljamo, da je vsota plačil ista stvar kot vrednost blaga ali dobiček.


In [ ]:
mapa = DATA / "trgovina"
narocila = pd.read_csv(mapa / "olist_orders_dataset.csv")
kupci = pd.read_csv(mapa / "olist_customers_dataset.csv", dtype={"customer_zip_code_prefix": "string"})
postavke = pd.read_csv(mapa / "olist_order_items_dataset.csv")
placila = pd.read_csv(mapa / "olist_order_payments_dataset.csv")
display(pd.DataFrame({
    "tabela": ["naročila", "kupci", "postavke", "plačila"],
    "vrstic": [len(narocila), len(kupci), len(postavke), len(placila)],
}))

## 2. Past združevanja: eno naročilo in več dogodkov

Izberemo pravo naročilo z več postavkami in več plačili. Neposredni merge ponovi
vsako postavko za vsako plačilo. Naročilo zato ni postalo več nakupov, zneski pa
se lahko večkrat seštejejo. `drop_duplicates()` ni splošna rešitev: lahko odstrani
veljavne, podobne postavke ali plačila.


In [ ]:
stevila = postavke.groupby("order_id").size().rename("postavke").to_frame()
stevila = stevila.join(placila.groupby("order_id").size().rename("placila"))
id_primera = stevila.loc[stevila["postavke"].gt(1) & stevila["placila"].gt(1)].index[0]
p = postavke.loc[postavke["order_id"].eq(id_primera), ["order_id", "order_item_id", "price"]]
l = placila.loc[placila["order_id"].eq(id_primera), ["order_id", "payment_sequential", "payment_value"]]
napacno = p.merge(l, on="order_id")
print(f"Postavk: {len(p)}; plačil: {len(l)}; vrstic po napačnem joinu: {len(napacno)}")
display(napacno[["order_item_id", "price", "payment_sequential", "payment_value"]])
display(pd.DataFrame({
    "pravilno": [p["price"].sum(), l["payment_value"].sum()],
    "po napačnem joinu": [napacno["price"].sum(), napacno["payment_value"].sum()],
}, index=["vrednost blaga", "vsota plačil"]))

## 3. Najprej agregiramo na raven naročila

Vsako podrobno tabelo najprej povzamemo po `order_id`, šele nato združimo z naročili.
`validate="one_to_one"` preveri kardinalnost. Left join ohrani tudi naročila brez
postavk ali plačil; neznanega zneska ne pretvorimo v ničlo. `min_count=1` pri vsoti
prepreči, da bi bil seštevek povsem manjkajočih cen prikazan kot 0.


In [ ]:
postavke_po_narocilu = postavke.groupby("order_id", as_index=False).agg(
    vrednost_narocila=("price", lambda s: s.sum(min_count=1)),
    postnina=("freight_value", lambda s: s.sum(min_count=1)),
    stevilo_postavk=("order_item_id", "size"),
    stevilo_prodajalcev=("seller_id", "nunique"),
)
placila_po_narocilu = placila.groupby("order_id", as_index=False).agg(
    vsota_placil=("payment_value", lambda s: s.sum(min_count=1)),
    stevilo_placil=("payment_sequential", "size"),
)
tabela_narocil = (
    narocila[["order_id", "customer_id", "order_purchase_timestamp"]]
    .merge(kupci[["customer_id", "customer_unique_id", "customer_state"]],
           on="customer_id", how="left", validate="one_to_one")
    .merge(postavke_po_narocilu, on="order_id", how="left", validate="one_to_one")
    .merge(placila_po_narocilu, on="order_id", how="left", validate="one_to_one")
)
assert tabela_narocil["order_id"].is_unique and len(tabela_narocil) == len(narocila)
assert tabela_narocil["customer_unique_id"].notna().all()
np.testing.assert_allclose(tabela_narocil["vrednost_narocila"].sum(), postavke["price"].sum())
np.testing.assert_allclose(tabela_narocil["vsota_placil"].sum(), placila["payment_value"].sum())
print(f"Naročil brez znane košarice: {tabela_narocil['vrednost_narocila'].isna().sum():,}")
print(f"Naročil brez znane vsote plačil: {tabela_narocil['vsota_placil'].isna().sum():,}")
display(tabela_narocil.head())

## 4. Najprej časovni dogovor, nato značilke

Presečni datum je **T = 2018-07-01**, zgodovina pa **180 dni**, ne šest koledarskih
mesecev. Vključimo `[T − 180 dni, T)`. Nakup na spodnji meji šteje, nakup ob T ne.
Iz časa nakupa izpeljemo dan v tednu, uro in kazalnik vikenda (`dayofweek >= 5`).
Časi v CSV-ju nimajo časovnega pasu; uporabimo zapisani čas. Pred dejansko uporabo
bi morali preveriti njegov pas in pravila beleženja.

Vključujemo zabeležena oddana naročila z znanimi postavkami. Ne filtriramo po končnem
statusu `delivered` in ne uporabljamo datumov dostave ali ocen, ki bi lahko nastali
po T. Gre za **retrospektiven učni prikaz**: statični izvoz ne vsebuje popolne
zgodovine sprememb ali časa, ko je bila vsaka sprememba zabeležena. Predpostavimo,
da so bili zapis nakupa, postavke in pripadajoča lokacija takrat dostopni; tega iz
izvoza ne moremo v celoti dokazati. Monetary ni potrjeni neto prihodek.


In [ ]:
T = pd.Timestamp("2018-07-01")
dolzina_okna_dni = 180
zacetek = T - pd.DateOffset(days=dolzina_okna_dni)
tabela_narocil["cas_nakupa"] = pd.to_datetime(tabela_narocil["order_purchase_timestamp"])
tabela_narocil["dan_v_tednu"] = tabela_narocil["cas_nakupa"].dt.dayofweek
tabela_narocil["ura_nakupa"] = tabela_narocil["cas_nakupa"].dt.hour
tabela_narocil["vikend"] = tabela_narocil["dan_v_tednu"].ge(5).astype(float).where(
    tabela_narocil["cas_nakupa"].notna()
)

v_oknu = tabela_narocil.loc[
    tabela_narocil["cas_nakupa"].ge(zacetek) & tabela_narocil["cas_nakupa"].lt(T)
].copy()
brez_kosarice = v_oknu["vrednost_narocila"].isna()
zgodovina = v_oknu.loc[~brez_kosarice].copy()
print(f"Okno: [{zacetek}, {T})")
print(f"Naročil v oknu: {len(v_oknu):,}; izpuščenih brez košarice: {brez_kosarice.sum():,}")
print(f"Naročil za RFM: {len(zgodovina):,}")
print("Kupcev samo z izpuščenimi naročili:",
      v_oknu["customer_unique_id"].nunique() - zgodovina["customer_unique_id"].nunique())
assert zgodovina["cas_nakupa"].ge(zacetek).all() and zgodovina["cas_nakupa"].lt(T).all()
display(zgodovina[["cas_nakupa", "dan_v_tednu", "ura_nakupa", "vikend"]].head())

## 5. Ena vrstica na kupca: RFM in smiselne razširitve

- **R:** koledarski dnevi od zadnjega nakupa do T.
- **F:** število različnih naročil v oknu, ne število postavk ali plačil.
- **M:** vsota vrednosti blaga v istem oknu, brez poštnine, v BRL.
- Povprečna in največja košarica ter delež vikend naročil so dodatni povzetki.

Kupcev brez vključenega naročila ni v tej tabeli; njihove R ne nastavimo na 0.
Recency računamo po **koledarskih dnevih**, ne kot del dneva v 24-urnih enotah.
Razmerja zaščitimo pred ničelnim imenovalcem, čeprav je F v tem naboru vsaj 1.


In [ ]:
rfm = zgodovina.groupby("customer_unique_id").agg(
    F=("order_id", "nunique"),
    M=("vrednost_narocila", "sum"),
    povprecna_kosarica=("vrednost_narocila", "mean"),
    najvecja_kosarica=("vrednost_narocila", "max"),
    zadnji_nakup=("cas_nakupa", "max"),
    vikend_narocila=("vikend", "sum"),
    stevilo_postavk=("stevilo_postavk", "sum"),
)
rfm["R"] = (T.normalize() - rfm["zadnji_nakup"].dt.normalize()).dt.days
rfm["delez_vikend_narocil"] = rfm["vikend_narocila"] / rfm["F"].where(rfm["F"].ne(0))

# Lokacija zadnjega zgodovinskega naročila, ne nujno stalna lastnost kupca.
zadnja_lokacija = (
    zgodovina.sort_values(["cas_nakupa", "order_id"], kind="stable")
    .drop_duplicates("customer_unique_id", keep="last")
    .set_index("customer_unique_id")["customer_state"]
)
rfm = rfm.join(zadnja_lokacija, validate="one_to_one")
rfm["razred_frekvence"] = pd.cut(
    rfm["F"], bins=[0, 1, 2, np.inf], labels=["1 nakup", "2 nakupa", "3 ali več"], ordered=True
)
assert rfm.index.is_unique
assert rfm["F"].sum() == zgodovina["order_id"].nunique()
np.testing.assert_allclose(rfm["M"].sum(), zgodovina["vrednost_narocila"].sum())
np.testing.assert_allclose(rfm["povprecna_kosarica"], rfm["M"] / rfm["F"])
assert rfm["R"].between(1, dolzina_okna_dni).all()
assert rfm["delez_vikend_narocil"].between(0, 1).all()
print(f"Kupcev: {len(rfm):,}; vsi ključi in skupni zneski so preverjeni.")
display(rfm[["R", "F", "M", "povprecna_kosarica", "delez_vikend_narocil", "customer_state"]].head())
display(rfm["F"].value_counts().sort_index().rename("kupci").to_frame())

In [ ]:
fig, osi = plt.subplots(1, 3, figsize=(13, 4))
osi[0].hist(rfm["R"], bins=30, edgecolor="white")
osi[0].set(xlabel="R: koledarski dnevi", ylabel="Kupci", title="Recency glede na fiksni T")
rfm["F"].value_counts().sort_index().plot.bar(ax=osi[1])
osi[1].set(xlabel="F: različna naročila", ylabel="Kupci", title="Frequency ni zelo raznolika")
osi[2].hist(rfm["M"], bins=50, edgecolor="white")
osi[2].set(xlabel="M: vrednost blaga (BRL)", ylabel="Kupci", title="Monetary v istem oknu")
fig.tight_layout()
plt.show()

**Za pogovor:** zakaj je večina F enaka 1? Zakaj ta RFM še ne zagotavlja dobrih
segmentov ali napovedi življenjske vrednosti? Kako bi se definicija spremenila,
če bi vključili poštnino, vračila ali drugo časovno okno?


## 6. Izberemo značilke in pojasnimo njihov pomen

Tabela `znacilke` vsebuje stolpce, ki jih bo model dobil kot vhod. Identifikator
kupca hranimo ločeno, da lahko rezultate pozneje povežemo z ustreznim kupcem.
Datum zadnjega nakupa smo že uporabili pri izračunu `R`; v tej vhodni tabeli
uporabimo `R`.

`customer_state` je brazilska zvezna država pri zadnjem naročilu v opazovanem
obdobju. Če imata naročili isti čas, za ponovljiv izbor uporabimo `order_id`.
Razred po številu naročil uporabimo za vajo ordinalnega kodiranja. Ker smo ga
izračunali iz `F`, z njim ne dodamo novih podatkov o kupcu.


In [ ]:
kljuci = rfm.index.to_series(name="customer_unique_id")
znacilke = rfm[["R", "F", "M", "povprecna_kosarica", "najvecja_kosarica",
                 "delez_vikend_narocil", "customer_state", "razred_frekvence"]].copy()
znacilke["kategorija_manjka"] = znacilke["customer_state"].isna().astype(int)
assert not np.isinf(znacilke.select_dtypes(include="number").to_numpy()).any()
display(pd.DataFrame([
    ["R", "kupec", "koledarski dnevi", "zadnji nakup do T"],
    ["F", "kupec", "število naročil", "180 dni pred T"],
    ["M", "kupec", "BRL brez poštnine", "180 dni pred T"],
    ["delez_vikend_narocil", "kupec", "delež 0–1", "180 dni pred T"],
    ["customer_state", "kupec", "nominalna oznaka", "zadnje vključeno naročilo"],
], columns=["značilka", "raven", "enota", "okno / pravilo"]))

## Kodiranje 1: najprej ločimo učne in testne podatke

V tabeli `znacilke` ena vrstica predstavlja enega kupca oziroma stranko.
Identifikator in morebitno ciljno vrednost hranimo ločeno od značilk.
80 % vrstic namenimo učenju, 20 % pa preverjanju. Ista stranka sme biti samo
v eni od obeh skupin. Ta delitev je namenjena prikazu postopka; način preverjanja
modela bomo podrobneje obravnavali pri modeliranju.

Vsoto nakupov ali število naročil izračunamo po vnaprej določenih pravilih.
Seznam kategorij, njihove pogostosti, mediano za nadomeščanje manjkajočih
vrednosti in parametre skaliranja pa določimo **samo na učnih podatkih**.
Na testnih nato uporabimo iste naučene nastavitve.


In [ ]:
kategoricni_stolpec = 'customer_state'
urejeni_stolpec = 'razred_frekvence'
urejene_kategorije = ['1 nakup', '2 nakupa', '3 ali več']
numericni_stolpci = ['R', 'F', 'M', 'povprecna_kosarica', 'najvecja_kosarica', 'delez_vikend_narocil', 'kategorija_manjka']

assert znacilke.index.is_unique
assert not {"Customer_ID", "customer_unique_id", "churn"}.intersection(znacilke.columns)
ucni_indeksi, testni_indeksi = train_test_split(znacilke.index, test_size=0.2, random_state=42)
X_train = znacilke.loc[ucni_indeksi].copy()
X_test = znacilke.loc[testni_indeksi].copy()
assert set(kljuci.loc[ucni_indeksi]).isdisjoint(set(kljuci.loc[testni_indeksi]))
print(f"Učnih strank: {len(X_train):,}; testnih: {len(X_test):,}")

## Kodiranje 2: redke kategorije v OTHER in referenčna kategorija

Regija nima naravnega vrstnega reda. Kategorije z manj kot **3 % učnih vrstic**
združimo v `OTHER`. Prag je izbran za prikaz; pogostosti ocenimo samo na učni
množici. Spremeni `prag_redkosti` in preveri, katere kategorije se združijo.

`OneHotEncoder(min_frequency=..., drop="first")` po združevanju izpusti prvo
ohranjeno kategorijo. Ta je **referenčna** in ima v one-hot bloku same ničle.
S tem odpravimo popolno kolinearnost tega bloka s konstantnim členom pri linearni
regresiji; izbiro kodiranja prilagodimo modelu.

`handle_unknown="infrequent_if_exist"` nove vrednosti usmeri v obstoječi `OTHER`.
Scikit-learn skupino interno imenuje `infrequent_sklearn`; v imenih izhodnih
stolpcev jo zaradi berljivosti prikažemo kot `OTHER`.

Manjkajoče vrednosti pred kodiranjem označimo `NEZNANO`. Če je ta oznaka redka ali
še nevidena, gre prav tako v `OTHER`; ločen `kategorija_manjka` ohrani razliko.
ID-jev in ciljnih oznak ne kodiramo kot vhode.


In [ ]:
prag_redkosti = 0.03  # Manj kot 3 % učnih vrstic -> OTHER.


def poimenuj_onehot(stolpec, kategorija):
    oznaka = "OTHER" if kategorija == "infrequent_sklearn" else kategorija
    return f"{stolpec}_{oznaka}"


kat_train = X_train[[kategoricni_stolpec]].astype("object").fillna("NEZNANO")
kat_test = X_test[[kategoricni_stolpec]].astype("object").fillna("NEZNANO")
enkoder = OneHotEncoder(
    min_frequency=prag_redkosti,
    drop="first",
    handle_unknown="infrequent_if_exist",
    sparse_output=False,
    feature_name_combiner=poimenuj_onehot,
)
ucni_onehot = enkoder.fit_transform(kat_train)
testni_onehot = enkoder.transform(kat_test)

redke = enkoder.infrequent_categories_[0]
redke_kategorije = [] if redke is None else redke.tolist()
referencna_kategorija = enkoder.categories_[0][int(enkoder.drop_idx_[0])]
referencna_skupina = (
    "OTHER" if referencna_kategorija in redke_kategorije else referencna_kategorija
)
pregled_kategorij = (
    kat_train[kategoricni_stolpec].value_counts()
    .rename_axis(kategoricni_stolpec).to_frame("ucnih_primerov")
)
pregled_kategorij["delez (%)"] = pregled_kategorij["ucnih_primerov"] / len(kat_train) * 100
pregled_kategorij["skupina"] = pregled_kategorij.index.map(
    lambda kategorija: "OTHER" if kategorija in redke_kategorije else kategorija
)
display(pregled_kategorij.round(2))
print("Redke učne kategorije -> OTHER:", redke_kategorije)
print("Referenčna skupina (same ničle):", referencna_skupina)

imena_onehot = enkoder.get_feature_names_out([kategoricni_stolpec])
train_onehot = pd.DataFrame(ucni_onehot, index=X_train.index, columns=imena_onehot)
test_onehot = pd.DataFrame(testni_onehot, index=X_test.index, columns=imena_onehot)
assert train_onehot.columns.equals(test_onehot.columns)
print("Oblika učne in testne matrike:", train_onehot.shape, test_onehot.shape)
display(pd.concat([kat_train.head(5), train_onehot.head(5)], axis=1))

## Kodiranje 3: referenčna, nova in manjkajoča kategorija

Naslednje tri vrstice so **umetni prikazi**, izpeljani iz kopij testnih značilk.
Prva ima referenčno kategorijo, druga novo regijo, tretja manjkajočo vrednost.
Ne dodajamo jih učnim podatkom in enkoderja ne učimo ponovno.

Pri privzetem primeru je `OTHER` nastal iz redkih učnih kategorij. Referenčna
kategorija dobi same ničle, nova pa 1 v stolpcu `OTHER`. Manjkajoč vhod še vedno
označuje tudi `kategorija_manjka = 1`.

**Robni primer:** če nobena učna kategorija ni redka, skupina `OTHER` ne nastane.
Nova vrednost tedaj dobi same ničle in je v tem bloku ne ločimo od reference.
Spodaj preverimo, ali stolpec `OTHER` obstaja; ne predpostavljamo ga pri vsakem
izbranem stolpcu ali pragu. Opozorilo knjižnice ob novi kategoriji je pri tem
preizkusu pričakovano; preverimo dejanski izhod kodiranja.


In [ ]:
nove_vrstice = X_test.head(3).copy()
nove_vrstice.index = [
    "umetna referenčna kategorija", "umetna nova kategorija", "umetna manjkajoča kategorija"
]
nove_vrstice[kategoricni_stolpec] = [referencna_kategorija, "NOVA_REGIJA_ZA_PRIMER", np.nan]
nove_vrstice["kategorija_manjka"] = nove_vrstice[kategoricni_stolpec].isna().astype(int)
novi_kategoriji = nove_vrstice[[kategoricni_stolpec]].astype("object").fillna("NEZNANO")
slovar_pred = enkoder.categories_[0].copy()
novi_onehot = pd.DataFrame(
    enkoder.transform(novi_kategoriji), index=nove_vrstice.index, columns=imena_onehot
)
np.testing.assert_array_equal(enkoder.categories_[0], slovar_pred)
assert novi_onehot.loc["umetna referenčna kategorija"].eq(0).all()

stolpec_other = f"{kategoricni_stolpec}_OTHER"
if stolpec_other in novi_onehot.columns:
    assert novi_onehot.loc["umetna nova kategorija", stolpec_other] == 1
    print("Nova vrednost gre v OTHER; referenčna kategorija ima same ničle.")
else:
    assert novi_onehot.loc["umetna nova kategorija"].eq(0).all()
    print("Ni ločenega stolpca OTHER: nova vrednost ima v tem bloku isto kodo kot referenca.")
display(pd.concat([novi_kategoriji, nove_vrstice[["kategorija_manjka"]], novi_onehot], axis=1))

## Kodiranje 4: kategorije z vrstnim redom pretvorimo v številčne kode

Iz števila naročil `F` smo oblikovali tri razrede. Njihov vrstni red je jasen:
en nakup, dva nakupa, nato trije ali več. Z `OrdinalEncoder` vsakemu razredu
dodelimo številčno kodo. Temu pravimo **ordinalno kodiranje**.

| Število naročil F | Razred | Koda |
|---|---|---:|
| 1 | `1 nakup` | 0 |
| 2 | `2 nakupa` | 1 |
| 3 ali več | `3 ali več` | 2 |

Kupec s petimi naročili dobi kodo **2**, ker spada v razred `3 ali več`.
Koda označuje razred. Dejansko število njegovih naročil je še vedno 5.
Vrstni red kod določimo s seznamom `urejene_kategorije`.

- `fit_transform()` uporabimo na učnih vrsticah, `transform()` pa na testnih,
  da za oboje velja isto kodiranje.
- Kategorija, ki je ni na našem seznamu, dobi **−1**. Spodaj to pokažemo z
  umetnim primerom. −1 pomeni »nepoznana kategorija«; ne pove števila naročil.
- Razlika med kodama 0 in 1 je enaka razliki med 1 in 2. Razliki med skupinami
  kupcev pa nista nujno enako veliki. Linearni model bi koraka obravnaval enako;
  po potrebi razrede kodiramo z one-hot.

Tukaj vadimo kodiranje. Razred je izpeljan iz `F` in izgubi podatek o točnem
številu naročil nad 2. Pri modeliranju lahko uporabimo kar `F`; sam razred še
ne pomeni boljše napovedi.


In [ ]:
ord_train = X_train[[urejeni_stolpec]].astype("object").fillna("NEZNANO")
ord_test = X_test[[urejeni_stolpec]].astype("object").fillna("NEZNANO")
ordinalni = OrdinalEncoder(
    categories=[urejene_kategorije], handle_unknown="use_encoded_value", unknown_value=-1
)
ucne_kode = ordinalni.fit_transform(ord_train)
testne_kode = ordinalni.transform(ord_test)
display(pd.DataFrame({"razred": ord_train.iloc[:8, 0], "koda": ucne_kode[:8, 0]}))
print("Vrstni red:", ordinalni.categories_[0].tolist())
print("Koda umetne nove stopnje:", ordinalni.transform(
    pd.DataFrame({urejeni_stolpec: ["NOVA_STOPNJA_ZA_PRIMER"]})
)[0, 0])